## Explanation for Cell 1: Dependency Installation

This first cell documents optional package installation.

Use it when:
- Running on a fresh environment.
- Missing one of the required libraries.

Packages used:
- `torch`: model training/inference.
- `datasets`: WikiText-2 loading.
- `transformers`: GPT-2 tokenizer.
- `tqdm`: progress bars.
- `pandas`: results table + CSV export.
- `wandb`: experiment tracking.
- `hydra-core` (+ built-in `omegaconf`): hierarchical YAML configs and override strings (assignment-recommended workflow).

Tip: Keep this cell commented in stable environments to avoid reinstall overhead.

In [ ]:
# Cell 1 — Install dependencies (uncomment if needed)
# !pip install -q torch datasets transformers tqdm pandas wandb hydra-core

## Explanation for Cell 2: Imports, Runtime Setup, and Reproducibility

This cell prepares the runtime environment.

What it covers:
- Imports PyTorch, datasets, tokenizer tools, progress bars, W&B, and analysis libs.
- Defines `set_seed(...)` to reduce random variation between runs.
- Selects compute device (GPU if available, otherwise CPU).
- Prints hardware info (useful for report reproducibility).

Why this matters:
- Comparable experiments need deterministic setup as much as possible.
- Hardware details are required context for throughput and latency claims.

In [4]:
# Cell 2 — Imports + device + seed
import os
import gc
import math
import time
import copy
import random
from dataclasses import dataclass, asdict, fields
from pathlib import Path
from typing import Optional
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm.auto import tqdm
import wandb

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USING_CUDA = device.type == "cuda"
print("Device:", device)
if USING_CUDA:
    print("GPU:", torch.cuda.get_device_name(0))


def cleanup_cuda(*objs):
    for obj in objs:
        del obj
    gc.collect()
    if USING_CUDA:
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def safe_cuda_empty_cache(reset_peak=False):
    """Best-effort CUDA cache cleanup that's safe on CPU-only runs."""
    gc.collect()
    if not USING_CUDA:
        return
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    if reset_peak:
        torch.cuda.reset_peak_memory_stats()


def gpu_mem_snapshot_mb():
    if not USING_CUDA:
        return {"allocated_mb": 0.0, "reserved_mb": 0.0, "max_allocated_mb": 0.0}
    return {
        "allocated_mb": torch.cuda.memory_allocated() / (1024 ** 2),
        "reserved_mb": torch.cuda.memory_reserved() / (1024 ** 2),
        "max_allocated_mb": torch.cuda.max_memory_allocated() / (1024 ** 2),
    }

Device: cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


## Explanation for Cell 3: Central Configuration Object

This dataclass is the single source of truth for experiment settings.

It includes:
- Dataset/tokenizer config.
- Model dimensions (`d_model`, heads, layers, FFN).
- Attention-specific knobs (`n_kv_heads`, `local_window_size`).
- Training hyperparameters.
- Evaluation cadence and limits.

Why this pattern helps:
- Easy reproducibility: one printed config defines the run.
- Easy ablations: change one parameter at a time.
- Compatible with W&B config logging via `asdict(cfg)`.

Plug-and-play workflow:
- Add new hyperparameters here when creating new attention variants.
- Read them inside your custom attention class constructor.

### Hydra (`conf/config.yaml`)
- Defaults live in **`conf/config.yaml`**. The notebook composes them with **`HYDRA_OVERRIDES`** (same syntax as `python train.py epochs=3`).
- Shell: `set HYDRA_OVERRIDES=epochs=2 attention_type=gqa` (Windows) then rerun the config cell.
- W&B project / entity / group / tags under YAML key **`wandb:`** — logged into every run and kept out of `TransformerLM` fields.
- Last composed config is saved to **`hydra_outputs/last_composed_config.yaml`** for the report / reproducibility.
- Hydra itself is not a metrics dashboard; it pairs with **W&B** (below) where you compare curves. Set **`USE_HYDRA = False`** in the config cell to fall back to dataclass defaults only.

### How to configure experiments (quick reference)
| Goal | What to change |
|------|----------------|
| Baseline Part 1 (1024 context) | `cfg.context_len = 1024`, `attention_type='scaled_dot_product'`, train long enough (`epochs`). |
| Attention swap | `cfg.attention_type` or pass a different first string to `run_single_experiment`. |
| Positional encoding | `cfg.positional_encoding_type` or `run_single_experiment(..., positional_encoding_type='rope')`. |
| GQA/MQA memory / speed | `n_heads` / `n_kv_heads` (use `mqa` or `gqa` with small `n_kv_heads`). |
| Local / sparse patterns | `local_window_size`, `sparse_block_size`. |
| Conv hybrids (Part 4) | `cfg.block_type` and `conv_kernel_size` (`conv_groups` for grouped conv). |
| Longer sequences than learned table | Raise `max_position_embeddings` **before** training. |
| Per-run overrides without editing `cfg` | `run_single_experiment(cfg, ..., attn, ctx, bs, block_type='conv_before_attention')`. |

We intentionally prioritize **512 + 1024 contexts** plus **more attention variants and head/KV ablations** instead of 2048/4096 sweeps (same reporting requirements; document this choice in your LaTeX report).

In [5]:
# Cell 3 — Config
@dataclass
class Config:
    dataset_name: str = "wikitext"
    dataset_config: str = "wikitext-2-raw-v1"
    tokenizer_name: str = "gpt2"
    vocab_size: int = 50257

    d_model: int = 512
    n_heads: int = 8
    n_kv_heads: int = 2
    n_layers: int = 8
    d_ff: int = 2048
    dropout: float = 0.1
    # Positional: learned | sinusoidal | none | rope | alibi | relative_shaw
    positional_encoding_type: str = "learned"
    # Blocks: standard_transformer_block | conv_before_attention |
    #          interleaved_conv_attention | alternate_attention_dwconv
    block_type: str = "standard_transformer_block"
    local_window_size: int = 256
    sparse_block_size: int = 64
    relative_max_distance: int = 128
    max_position_embeddings: int = 8192
    conv_kernel_size: int = 7
    conv_groups: int = 1
    alibi_slope_scale: float = 1.0

    context_len: int = 1024
    batch_size: int = 4
    epochs: int = 1
    lr: float = 3e-4
    weight_decay: float = 0.1
    grad_clip: float = 1.0
    eval_every_steps: int = 250
    max_eval_batches: Optional[int] = 100

    # Runtime policy
    use_amp: bool = True
    amp_dtype: str = "fp16"  # fp16 | bf16

    attention_type: str = "scaled_dot_product"
    seed: int = 42


from omegaconf import OmegaConf
from hydra import compose, initialize_config_dir

_HYDRA_OMEGA = None


def _config_field_names():
    return {f.name for f in fields(Config)}


def config_and_wandb_from_hydra(overrides):
    """Compose conf/config.yaml; split wandb:* from fields consumed by Config / TransformerLM."""
    global _HYDRA_OMEGA
    conf_dir = Path.cwd() / "conf"
    if not conf_dir.is_dir():
        raise FileNotFoundError(
            f"Expected Hydra configs at {conf_dir}. Run with notebook cwd = repo root (AI-Research)."
        )
    with initialize_config_dir(version_base=None, config_dir=str(conf_dir.resolve())):
        _HYDRA_OMEGA = compose(config_name="config", overrides=list(overrides or []))
    flat = OmegaConf.to_container(_HYDRA_OMEGA, resolve=True)
    wb = dict(flat.pop("wandb", {}))
    cfg_dict = {k: v for k, v in flat.items() if k in _config_field_names()}
    out_dir = Path.cwd() / "hydra_outputs"
    out_dir.mkdir(parents=True, exist_ok=True)
    OmegaConf.save(_HYDRA_OMEGA, out_dir / "last_composed_config.yaml")
    return Config(**cfg_dict), wb


USE_HYDRA = True
# Same syntax as Hydra CLI, e.g. ["epochs=3", "attention_type=gqa", "wandb.group=sweep_oct"]
HYDRA_OVERRIDES = []
_env_ov = os.environ.get("HYDRA_OVERRIDES", "").strip()
if _env_ov:
    HYDRA_OVERRIDES = _env_ov.split()

if USE_HYDRA:
    cfg, _WANDB_YAML = config_and_wandb_from_hydra(HYDRA_OVERRIDES)
else:
    _HYDRA_OMEGA = None
    cfg = Config()
    _WANDB_YAML = {"project": "wikitext2-attention-variants", "entity": None, "group": None, "tags": []}

if not USING_CUDA:
    cfg.use_amp = False
if cfg.amp_dtype not in ("fp16", "bf16"):
    raise ValueError("amp_dtype must be 'fp16' or 'bf16'")

print(cfg)
print("Hydra wandb:", _WANDB_YAML)
print(f"Runtime policy -> CUDA: {USING_CUDA}, AMP: {cfg.use_amp}, amp_dtype: {cfg.amp_dtype}")

Config(dataset_name='wikitext', dataset_config='wikitext-2-raw-v1', tokenizer_name='gpt2', vocab_size=50257, d_model=512, n_heads=8, n_kv_heads=2, n_layers=8, d_ff=2048, dropout=0.1, positional_encoding_type='learned', block_type='standard_transformer_block', local_window_size=256, sparse_block_size=64, relative_max_distance=128, max_position_embeddings=8192, conv_kernel_size=7, conv_groups=1, alibi_slope_scale=1.0, context_len=1024, batch_size=4, epochs=1, lr=0.0003, weight_decay=0.1, grad_clip=1.0, eval_every_steps=250, max_eval_batches=100, use_amp=True, amp_dtype='fp16', attention_type='scaled_dot_product', seed=42)
Hydra wandb: {'project': 'wikitext2-attention-hydra', 'entity': None, 'group': 'saidl_core_ml', 'tags': ['hydra', 'saidl-core-ml']}
Runtime policy -> CUDA: True, AMP: True, amp_dtype: fp16


## Pre-run sanity check (device + memory)

Run this once before sweeps. It:
- prints runtime policy (CUDA + AMP)
- prints GPU memory before cleanup
- forces cache cleanup
- prints GPU memory after cleanup

In [6]:
# Pre-run sanity cell — Attention notebook
print("=== Runtime policy ===")
print("Device:", device)
print("USING_CUDA:", USING_CUDA)
print("AMP enabled by config:", getattr(cfg, "use_amp", None))
print("AMP dtype:", getattr(cfg, "amp_dtype", None))

if USING_CUDA:
    before = gpu_mem_snapshot_mb()
    print(f"Before cleanup -> allocated={before['allocated_mb']:.1f} MB, reserved={before['reserved_mb']:.1f} MB, peak={before['max_allocated_mb']:.1f} MB")

safe_cuda_empty_cache(reset_peak=True)

if USING_CUDA:
    after = gpu_mem_snapshot_mb()
    print(f"After cleanup  -> allocated={after['allocated_mb']:.1f} MB, reserved={after['reserved_mb']:.1f} MB, peak={after['max_allocated_mb']:.1f} MB")
else:
    print("CPU mode: no CUDA memory to clear.")

print("Sanity check complete. Safe to start runs.")

=== Runtime policy ===
Device: cuda
USING_CUDA: True
AMP enabled by config: True
AMP dtype: fp16
Before cleanup -> allocated=0.0 MB, reserved=0.0 MB, peak=0.0 MB
After cleanup  -> allocated=0.0 MB, reserved=0.0 MB, peak=0.0 MB
Sanity check complete. Safe to start runs.


## Explanation for Cell 4: Weights & Biases + Hydra wiring

**Hydra** owns structured defaults in `conf/config.yaml` (including nested `wandb:`). There is no separate Hydra “metrics dashboard”; Hydra handles configuration composition and overrides; **W&B** is where you visualize loss/perplexity/latency across runs.

This cell:
- Authenticates (`wandb.login()`).
- Reads **`WANDB_PROJECT`**, **`WANDB_ENTITY`**, **`WANDB_GROUP`**, **`WANDB_EXTRA_TAGS`** from the composed `_WANDB_YAML` populated in the Hydra config cell (falls back if `USE_HYDRA = False`).

**Same W&B project vs new project**
- Keeping **`wikitext2-attention-variants`** is fine: runs accumulate; use **groups** (`wandb.group`) and **tags** to separate phases.
- Starting **`wikitext2-attention-hydra`** (default in YAML) gives a clean slate for Hydra-driven runs without deleting history.
- You can override at compose time: `wandb.project=wikitext2-attention-variants` in `HYDRA_OVERRIDES`.

Best practices:
- One mental “study” per **project** or per **group** inside one project.
- Log full Hydra compositions into `wandb.config` (done automatically when Hydra is enabled).

In [7]:
# Cell 4 — W&B setup (project/group/tags from Hydra `wandb:` if USE_HYDRA)
# Option A: os.environ["WANDB_API_KEY"] = "..."
# Option B: interactive login
wandb.login()

WANDB_PROJECT = _WANDB_YAML.get("project", "wikitext2-attention-variants")
WANDB_ENTITY = _WANDB_YAML.get("entity")
WANDB_GROUP = _WANDB_YAML.get("group") or None
WANDB_EXTRA_TAGS = list(_WANDB_YAML.get("tags") or [])

print("W&B → project:", WANDB_PROJECT, "| entity:", WANDB_ENTITY, "| group:", WANDB_GROUP, "| extra tags:", WANDB_EXTRA_TAGS)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\sidsr\_netrc.
wandb: Currently logged in as: siddharthsrinivasan2002 (siddharthsrinivasan2002-national-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B → project: wikitext2-attention-hydra | entity: None | group: saidl_core_ml | extra tags: ['hydra', 'saidl-core-ml']


## Explanation for Cell 5: Data Pipeline for WikiText-2

This cell creates the complete text-to-training-sample pipeline.

What it does:
- Loads WikiText-2 raw split.
- Uses GPT-2 tokenizer to convert text into token IDs.
- Concatenates documents with EOS separators into long token streams.
- Defines `LanguageModelingDataset` that returns:
  - `x = tokens[t : t+L]`
  - `y = tokens[t+1 : t+L+1]`
- Defines `build_dataloaders(...)` so context length/batch size can be changed per experiment.

Why this design is useful:
- The exact same data code works for any `context_len` (here we standardize on **512** and **1024** for throughput).
- You can fairly compare attention variants because only model attention changes, not data logic.

Important assumption:
- This uses fixed non-overlapping chunks (fast and simple). You can later switch to overlapping/random windows if needed.

Helpers:
- `build_valid_loader(...)` builds validation loaders only — used when evaluating trained checkpoints at **different sequence lengths** than training (extrapolation cell).

In [8]:
# Cell 5 — Data loading + dataset builders
raw_ds = load_dataset(cfg.dataset_name, cfg.dataset_config)
tokenizer = AutoTokenizer.from_pretrained(cfg.tokenizer_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_split(text_list):
    merged = tokenizer.eos_token.join(text_list)
    ids = tokenizer(merged, return_attention_mask=False, add_special_tokens=False)["input_ids"]
    return torch.tensor(ids, dtype=torch.long)

train_ids = tokenize_split(raw_ds["train"]["text"])
valid_ids = tokenize_split(raw_ds["validation"]["text"])
print("Train tokens:", train_ids.numel())
print("Valid tokens:", valid_ids.numel())

class LanguageModelingDataset(Dataset):      #here dataset comes from the Torch.utils.data import
    def __init__(self, token_ids: torch.Tensor, context_len: int):
        self.token_ids = token_ids
        self.context_len = context_len
        self.n_samples = (len(token_ids) - 1) // context_len

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        s = idx * self.context_len
        e = s + self.context_len
        x = self.token_ids[s:e]
        y = self.token_ids[s + 1:e + 1]
        return x, y

def build_dataloaders(train_ids, valid_ids, context_len, batch_size):
    train_ds = LanguageModelingDataset(train_ids, context_len)
    valid_ds = LanguageModelingDataset(valid_ids, context_len)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
    valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False, drop_last=False)
    return train_loader, valid_loader


def build_valid_loader(valid_ids, context_len, batch_size):
    """Validation-only loader (for extrapolation eval at L_test != training context)."""
    valid_ds = LanguageModelingDataset(valid_ids, context_len)
    return DataLoader(valid_ds, batch_size=batch_size, shuffle=False, drop_last=False)

Token indices sequence length is longer than the specified maximum sequence length for this model (2428601 > 1024). Running this sequence through the model will result in indexing errors


Train tokens: 2428601
Valid tokens: 251048


## Explanation for Cell 6: Positional Encodings, Attention Variants, Conv Hybrids (SAiDL Core ML)

This cell defines everything that must stay swappable per the assignment: attention mechanism, positional encoding family, and macro block layout.

### Positional encoding (`cfg.positional_encoding_type`)
| Key | Mechanism |
|-----|-----------|
| `learned` | Adds learned embedding table up to `cfg.max_position_embeddings`. |
| `sinusoidal` | Vaswani sinusoidal additive PE (bounded by same max length). |
| `none` | No additive PE (use with variants that bake position elsewhere). |
| `rope` | Rotary embeddings applied **inside** Q/K projections (additive PE is identity). Requires **even** head dimension. |
| `alibi` | Attention-with-linear-biases slopes added to attention logits (additive PE is identity). |
| `relative_shaw` | Learned bias by causal relative distance (clipped to `relative_max_distance`). |

Configure extrapolation length via `max_position_embeddings` (must be ≥ any evaluation sequence length).

### Attention (`cfg.attention_type` → `ATTENTION_REGISTRY`)
1. **`scaled_dot_product`** — baseline softmax causal attention.
2. **`local_window`** — sliding/restricted local causal attention (`local_window_size`).
3. **`sparse_block`** — block-sparse causal attention (`sparse_block_size` tokens per block).
4. **`linear_performer`** — causal kernel linear attention \(\phi=\mathrm{elu}+1\). Implemented with a **short loop over time only** (batch×heads vectorized); avoid old nested batch×time Python loops.
5. **`gqa`** — grouped-query attention (`n_heads` must divide by `n_kv_heads`).
6. **`mqa`** — multi-query attention wrapper (forces `n_kv_heads = 1`).
7. **`softmax_free_relu`** — ReLU² weights + renorm instead of softmax.

RoPE / ALiBi / Shaw biases are wired where meaningful for softmax-style attentions; `linear_performer` focuses on efficiency (assignment “linear attention” option).

### Block layouts (`cfg.block_type`)
- **`standard_transformer_block`** — Pre-LN + attention + FFN (default).
- **`conv_before_attention`** — causal Conv1d (`conv_kernel_size`, optional `conv_groups`) before attention (assignment §4 design).
- **`interleaved_conv_attention`** — alternating standard block and conv-before-attention block.
- **`alternate_attention_dwconv`** — even layers attention, odd layers depthwise-conv substitute (subset replacement).

### Adding a new attention module
1. Implement `forward(self, x)` → `(B, T, D)` with causal LM semantics.
2. Register in `ATTENTION_REGISTRY`.
3. Run via `run_single_experiment(..., "your_key", ctx, bs)` or add to sweep lists.

Implementation mirror file (optional editing aid outside Jupyter): `saidl_nb_cell06_architecture.py` — kept in repo next to this notebook.

In [9]:
# Cell 6 — Positional encodings + attention variants + model (SAiDL Core ML)
# Injected into SAIDL_BPGC_AttentionVariants_WandB.ipynb — keep in sync when editing.

import copy
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


class IdentityPositionalEncoding(nn.Module):
    def forward(self, x):
        return x


class LearnedPositionalEmbedding(nn.Module):
    def __init__(self, max_len: int, d_model: int):
        super().__init__()
        self.pos_emb = nn.Embedding(max_len, d_model)

    def forward(self, x):
        T = x.shape[1]
        pos = torch.arange(T, device=x.device).unsqueeze(0)
        return x + self.pos_emb(pos)


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, max_len: int, d_model: int):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x):
        return x + self.pe[:, : x.size(1), :]


def _rotate_half(x):
    d = x.shape[-1] // 2
    x1, x2 = x[..., :d], x[..., d:]
    return torch.cat((-x2, x1), dim=-1)


class RotaryEmbedding(nn.Module):
    """RoPE frequencies — applied inside attention to Q/K."""

    def __init__(self, dim: int, max_position_embeddings: int = 8192, base: float = 10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.max_position_embeddings = max_position_embeddings

    def forward(self, q, k):
        """q,k: (B, H, T, Dh); Dh must be even."""
        B, H, T, Dh = q.shape
        if Dh % 2 != 0:
            raise ValueError(f"RoPE requires even head dim, got Dh={Dh}")

        t = torch.arange(T, device=q.device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)  # (T, Dh/2)

        # Expand from Dh/2 to Dh so broadcasting matches q/k.
        cos = torch.repeat_interleave(freqs.cos(), repeats=2, dim=-1)[None, None, :, :]
        sin = torch.repeat_interleave(freqs.sin(), repeats=2, dim=-1)[None, None, :, :]

        q_embed = (q * cos) + (_rotate_half(q) * sin)
        k_embed = (k * cos) + (_rotate_half(k) * sin)
        return q_embed, k_embed


def causal_mask(T, device):
    return torch.triu(torch.ones(T, T, device=device, dtype=torch.bool), diagonal=1)


def build_alibi_slopes(n_heads: int, device, dtype):
    """Geometric slopes per head (Press et al., ALiBi)."""
    closest_power_of_2 = 2 ** math.floor(math.log2(n_heads))
    slopes = torch.pow(2.0, -torch.arange(0, closest_power_of_2, dtype=dtype, device=device) / closest_power_of_2)
    if closest_power_of_2 != n_heads:
        extra = torch.pow(2.0, -torch.arange(1, 2 * (n_heads - closest_power_of_2) + 1, 2, dtype=dtype, device=device) / closest_power_of_2)
        slopes = torch.cat([slopes, extra], dim=0)
    slopes = slopes[:n_heads].view(n_heads, 1, 1)
    return slopes


def alibi_bias(n_heads: int, T: int, device, dtype):
    """Additive causal bias (H, T, T): -slope * (i-j) for j<=i."""
    slopes = build_alibi_slopes(n_heads, device, dtype)
    pos = torch.arange(T, device=device)
    dist = pos.unsqueeze(0) - pos.unsqueeze(1)
    causal = torch.triu(torch.ones(T, T, device=device, dtype=torch.bool), diagonal=1)
    dist = dist.masked_fill(causal, 0)
    bias = -slopes * dist.unsqueeze(0).to(dtype)
    bias = bias.masked_fill(causal.unsqueeze(0), float("-inf"))
    return bias


class RelativePositionBias(nn.Module):
    """Learned bias buckets by relative distance (causal), Shaw-style simplified."""

    def __init__(self, n_heads: int, max_distance: int):
        super().__init__()
        self.n_heads = n_heads
        self.max_distance = max_distance
        self.bias = nn.Parameter(torch.zeros(n_heads, max_distance))

    def forward(self, T: int, device):
        idx = torch.arange(T, device=device)
        diff = idx.unsqueeze(0) - idx.unsqueeze(1)
        diff = diff.clamp(0, self.max_distance - 1)
        causal = torch.triu(torch.ones(T, T, device=device, dtype=torch.bool), diagonal=1)
        b = self.bias[:, diff]
        b = b.masked_fill(causal.unsqueeze(0), 0.0)
        return b


def pos_family(cfg) -> str:
    """additive | rope | alibi | relative"""
    pe = cfg.positional_encoding_type
    if pe in ("rope",):
        return "rope"
    if pe in ("alibi",):
        return "alibi"
    if pe in ("relative_shaw",):
        return "relative"
    return "additive"


class ScaledDotProductCausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0
        self.cfg = cfg
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        fam = pos_family(cfg)
        if fam == "rope":
            assert self.head_dim % 2 == 0, "RoPE requires even head_dim"
            self.rotary = RotaryEmbedding(self.head_dim, cfg.max_position_embeddings)
        else:
            self.rotary = None
        if fam == "relative":
            self.rel_bias = RelativePositionBias(cfg.n_heads, cfg.relative_max_distance)
        else:
            self.rel_bias = None
        self.qkv_proj = nn.Linear(cfg.d_model, 3 * cfg.d_model)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv_proj(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        cfg = self.cfg
        fam = pos_family(cfg)

        if fam == "rope":
            q, k = self.rotary(q, k)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(causal_mask(T, x.device), float("-inf"))
        if fam == "alibi":
            scores = scores + alibi_bias(self.n_heads, T, x.device, scores.dtype) * getattr(cfg, "alibi_slope_scale", 1.0)
        if fam == "relative":
            scores = scores + self.rel_bias(T, x.device).unsqueeze(0)
        w = self.dropout(F.softmax(scores, dim=-1))
        out = w @ v
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.out_proj(out)


class LocalWindowCausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0
        self.cfg = cfg
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.window = cfg.local_window_size
        fam = pos_family(cfg)
        if fam == "rope":
            assert self.head_dim % 2 == 0
            self.rotary = RotaryEmbedding(self.head_dim, cfg.max_position_embeddings)
        else:
            self.rotary = None
        if fam == "relative":
            self.rel_bias = RelativePositionBias(cfg.n_heads, cfg.relative_max_distance)
        else:
            self.rel_bias = None
        self.qkv_proj = nn.Linear(cfg.d_model, 3 * cfg.d_model)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv_proj(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        cfg = self.cfg
        fam = pos_family(cfg)
        if fam == "rope":
            q, k = self.rotary(q, k)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        i = torch.arange(T, device=x.device).unsqueeze(1)
        j = torch.arange(T, device=x.device).unsqueeze(0)
        dist = i - j
        invalid = (dist < 0) | (dist >= self.window)
        scores = scores.masked_fill(invalid, float("-inf"))
        if fam == "alibi":
            scores = scores + alibi_bias(self.n_heads, T, x.device, scores.dtype) * getattr(cfg, "alibi_slope_scale", 1.0)
        if fam == "relative":
            scores = scores + self.rel_bias(T, x.device).unsqueeze(0)
        w = self.dropout(F.softmax(scores, dim=-1))
        out = w @ v
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.out_proj(out)


class BlockSparseCausalSelfAttention(nn.Module):
    """Within each block of size Bk, full causal attention; no cross-block attention."""

    def __init__(self, cfg):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0
        self.cfg = cfg
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.block_size = cfg.sparse_block_size
        fam = pos_family(cfg)
        if fam == "rope":
            assert self.head_dim % 2 == 0
            self.rotary = RotaryEmbedding(self.head_dim, cfg.max_position_embeddings)
        else:
            self.rotary = None
        if fam == "relative":
            self.rel_bias = RelativePositionBias(cfg.n_heads, cfg.relative_max_distance)
        else:
            self.rel_bias = None
        self.qkv_proj = nn.Linear(cfg.d_model, 3 * cfg.d_model)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        B, T, D = x.shape
        bk = self.block_size
        q, k, v = self.qkv_proj(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        cfg = self.cfg
        fam = pos_family(cfg)
        if fam == "rope":
            q, k = self.rotary(q, k)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        cm = causal_mask(T, x.device)
        bi = torch.arange(T, device=x.device).unsqueeze(1) // bk
        bj = torch.arange(T, device=x.device).unsqueeze(0) // bk
        block_invalid = bi != bj
        invalid = cm | block_invalid
        scores = scores.masked_fill(invalid, float("-inf"))
        if fam == "alibi":
            scores = scores + alibi_bias(self.n_heads, T, x.device, scores.dtype) * getattr(cfg, "alibi_slope_scale", 1.0)
        if fam == "relative":
            scores = scores + self.rel_bias(T, x.device).unsqueeze(0)
        w = self.dropout(F.softmax(scores, dim=-1))
        out = w @ v
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.out_proj(out)


class LinearPerformerAttention(nn.Module):
    """Causal kernel linear attention phi(x)=elu(x)+1.

    Recurrence is O(T) steps with O(D^2) matmuls; batch and heads are vectorized.
    (Nested Python loops over batch x time caused multi-hour epochs.)
    """

    def __init__(self, cfg):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.k_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.v_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        B, T, D = x.shape
        H, Dh = self.n_heads, self.head_dim
        q = self.q_proj(x).view(B, T, H, Dh).transpose(1, 2)
        k = self.k_proj(x).view(B, T, H, Dh).transpose(1, 2)
        v = self.v_proj(x).view(B, T, H, Dh).transpose(1, 2)
        q = F.elu(q) + 1.0
        k = F.elu(k) + 1.0
        acc_kv = torch.zeros(B, H, Dh, Dh, device=x.device, dtype=x.dtype)
        acc_k = torch.zeros(B, H, Dh, device=x.device, dtype=x.dtype)
        steps = []
        for t in range(T):
            kt = k[:, :, t, :]
            vt = v[:, :, t, :]
            acc_kv = acc_kv + kt.unsqueeze(-1) * vt.unsqueeze(-2)
            acc_k = acc_k + kt
            qt = q[:, :, t, :]
            num = torch.matmul(acc_kv, qt.unsqueeze(-1)).squeeze(-1)
            den = (qt * acc_k).sum(dim=-1, keepdim=True).clamp(min=1e-6)
            steps.append((num / den).unsqueeze(2))
        out = torch.cat(steps, dim=2).transpose(1, 2).contiguous().view(B, T, D)
        out = self.dropout(out)
        return self.out_proj(out)


class GQACausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0
        assert cfg.n_heads % cfg.n_kv_heads == 0
        self.cfg = cfg
        self.n_heads = cfg.n_heads
        self.n_kv_heads = cfg.n_kv_heads
        self.group_size = cfg.n_heads // cfg.n_kv_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        fam = pos_family(cfg)
        if fam == "rope":
            assert self.head_dim % 2 == 0
            self.rotary = RotaryEmbedding(self.head_dim, cfg.max_position_embeddings)
        else:
            self.rotary = None
        if fam == "relative":
            self.rel_bias = RelativePositionBias(cfg.n_heads, cfg.relative_max_distance)
        else:
            self.rel_bias = None
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_heads * self.head_dim)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        B, T, D = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        k = k.repeat_interleave(self.group_size, dim=1)
        v = v.repeat_interleave(self.group_size, dim=1)
        cfg = self.cfg
        fam = pos_family(cfg)
        if fam == "rope":
            q, k = self.rotary(q, k)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(causal_mask(T, x.device), float("-inf"))
        if fam == "alibi":
            scores = scores + alibi_bias(self.n_heads, T, x.device, scores.dtype) * getattr(cfg, "alibi_slope_scale", 1.0)
        if fam == "relative":
            scores = scores + self.rel_bias(T, x.device).unsqueeze(0)
        w = self.dropout(F.softmax(scores, dim=-1))
        out = w @ v
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.out_proj(out)


class MQAttention(nn.Module):
    """Multi-query attention — single KV head (MQA paper)."""

    def __init__(self, cfg):
        super().__init__()
        mc = copy.deepcopy(cfg)
        mc.n_kv_heads = 1
        self.inner = GQACausalSelfAttention(mc)

    def forward(self, x):
        return self.inner(x)


class SoftmaxFreeReluAttention(nn.Module):
    def __init__(self, cfg, eps=1e-6):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0
        self.cfg = cfg
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.eps = eps
        fam = pos_family(cfg)
        if fam == "rope":
            assert self.head_dim % 2 == 0
            self.rotary = RotaryEmbedding(self.head_dim, cfg.max_position_embeddings)
        else:
            self.rotary = None
        self.qkv_proj = nn.Linear(cfg.d_model, 3 * cfg.d_model)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv_proj(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        if pos_family(self.cfg) == "rope":
            q, k = self.rotary(q, k)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(causal_mask(T, x.device), 0.0)
        w = F.relu(scores).pow(2)
        w = self.dropout(w)
        w = w / (w.sum(dim=-1, keepdim=True) + self.eps)
        out = w @ v
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.out_proj(out)


ATTENTION_REGISTRY = {
    "scaled_dot_product": ScaledDotProductCausalSelfAttention,
    "local_window": LocalWindowCausalSelfAttention,
    "sparse_block": BlockSparseCausalSelfAttention,
    "linear_performer": LinearPerformerAttention,
    "gqa": GQACausalSelfAttention,
    "mqa": MQAttention,
    "softmax_free_relu": SoftmaxFreeReluAttention,
}


class CausalConv1d(nn.Module):
    """Depthwise-separable friendly causal Conv1d over sequence."""

    def __init__(self, d_model: int, kernel_size: int, groups: int = 1):
        super().__init__()
        self.kernel_size = kernel_size
        self.conv = nn.Conv1d(d_model, d_model, kernel_size, groups=groups)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = F.pad(x, (self.kernel_size - 1, 0))
        x = self.conv(x).transpose(1, 2)
        return x


class StandardTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = ATTENTION_REGISTRY[cfg.attention_type](cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ff = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_ff),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_ff, cfg.d_model),
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class ConvBeforeAttentionBlock(nn.Module):
    """Hybrid (assignment §4): causal Conv1d before attention."""

    def __init__(self, cfg):
        super().__init__()
        self.conv = CausalConv1d(cfg.d_model, cfg.conv_kernel_size, groups=getattr(cfg, "conv_groups", 1))
        self.ln0 = nn.LayerNorm(cfg.d_model)
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = ATTENTION_REGISTRY[cfg.attention_type](cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ff = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_ff),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_ff, cfg.d_model),
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x):
        x = x + self.conv(self.ln0(x))
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class DepthwiseConvOnlyBlock(nn.Module):
    """Replace attention with causal depthwise conv + pointwise (subset replacement idea)."""

    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        dw = CausalConv1d(cfg.d_model, cfg.conv_kernel_size, groups=cfg.d_model)
        pw = nn.Linear(cfg.d_model, cfg.d_model)
        self.local = nn.Sequential(dw, pw)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ff = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_ff),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_ff, cfg.d_model),
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x):
        x = x + self.local(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


def build_blocks(cfg):
    n = cfg.n_layers
    bt = cfg.block_type
    blocks = []
    if bt == "standard_transformer_block":
        blocks = [StandardTransformerBlock(cfg) for _ in range(n)]
    elif bt == "conv_before_attention":
        blocks = [ConvBeforeAttentionBlock(cfg) for _ in range(n)]
    elif bt == "interleaved_conv_attention":
        for i in range(n):
            blocks.append(StandardTransformerBlock(cfg) if i % 2 == 0 else ConvBeforeAttentionBlock(cfg))
    elif bt == "alternate_attention_dwconv":
        for i in range(n):
            blocks.append(StandardTransformerBlock(cfg) if i % 2 == 0 else DepthwiseConvOnlyBlock(cfg))
    else:
        raise ValueError(f"Unknown block_type {bt}")
    return nn.ModuleList(blocks)


POSITIONAL_ENCODING_REGISTRY = {
    "learned": LearnedPositionalEmbedding,
    "sinusoidal": SinusoidalPositionalEncoding,
    "none": lambda max_len, d_model: IdentityPositionalEncoding(),
    "rope": lambda max_len, d_model: IdentityPositionalEncoding(),
    "alibi": lambda max_len, d_model: IdentityPositionalEncoding(),
    "relative_shaw": lambda max_len, d_model: IdentityPositionalEncoding(),
}


class TransformerLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        max_pos = cfg.max_position_embeddings
        pos_cls = POSITIONAL_ENCODING_REGISTRY[cfg.positional_encoding_type]
        self.pos_enc = pos_cls(max_pos, cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)
        self.blocks = build_blocks(cfg)
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight

    def forward(self, idx, targets=None):
        x = self.dropout(self.pos_enc(self.tok_emb(idx)))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss


## Explanation for Cell 7: Evaluation and Inference Latency

This cell defines two measurement helpers used by every experiment.

### `evaluate(...)`
- Runs model in `eval()` mode (dropout off).
- Computes average validation cross-entropy loss.
- Converts loss to perplexity using `exp(loss)`.

Why perplexity?
- It is the standard LM quality metric: lower is better.
- It directly answers assignment requirement (validation perplexity).

### `inference_latency(...)`
- Simulates autoregressive generation token-by-token.
- Includes warmup steps to avoid first-call timing bias.
- Uses CUDA synchronization for accurate GPU timing.
- Returns both:
  - tokens/sec (higher is better)
  - ms/token (lower is better)

Plug-and-play note:
- If your attention design has different compute complexity, latency effects will show up here clearly.

In [10]:
# Cell 7 — Eval and latency helpers

def safe_cuda_empty_cache(reset_peak=False):
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
            if reset_peak:
                torch.cuda.reset_peak_memory_stats()
        except RuntimeError:
            pass


@torch.no_grad()
def evaluate(model, loader, max_batches=None):
    model.eval()
    losses = []
    for i, (x, y) in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        _, loss = model(x, y)
        losses.append(loss.item())
    mean_loss = float(np.mean(losses))
    return mean_loss, math.exp(mean_loss)

@torch.no_grad()
def inference_latency(model, vocab_size, context_len, batch_size=1, gen_tokens=64, warmup=5):
    model.eval()
    idx = torch.randint(0, vocab_size, (batch_size, min(128, context_len)), device=device)
    for _ in range(warmup):
        logits, _ = model(idx, None)
        nxt = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
        idx = torch.cat([idx, nxt], dim=1)[:, -context_len:]
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(gen_tokens):
        logits, _ = model(idx, None)
        nxt = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
        idx = torch.cat([idx, nxt], dim=1)[:, -context_len:]
    if device.type == "cuda":
        torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    total_tokens = gen_tokens * batch_size
    return total_tokens / dt, (dt / total_tokens) * 1000.0

## Explanation for Cell 8: One Run = One Config (with W&B Logging)

This function executes a **single experimental configuration**: one attention type, one context length, one batch size.

Why this abstraction matters:
- Keeps each run reproducible and traceable.
- Makes sweeps simple (just call this function repeatedly).
- Centralizes logging so all runs are consistent.

What happens inside:
1. Clone base config and inject experiment-specific values.
2. Build dataloaders for the selected context length.
3. Instantiate model + optimizer.
4. Start a W&B run and log metadata (`num_parameters`, GPU, config).
5. Train for configured epochs while logging:
   - step loss
   - gradient norm
   - learning rate
   - tokens seen
6. Periodically evaluate on validation split.
7. Compute final metrics:
   - validation perplexity
   - train tokens/sec
   - inference tokens/sec and ms/token
   - peak GPU memory
   - stability diagnostics (`nan_steps`, gradient stats, loss std)
8. Push summary to W&B and close the run.

Plug-and-play note:
- If your attention module changes runtime behavior, this function captures that automatically through throughput/memory/latency metrics.

**Overrides:** Any `Config` field can be passed as a keyword after the batch size, e.g. `positional_encoding_type='rope'`, `n_heads=16`, `block_type='conv_before_attention'`.

**`return_model=True`:** Returns `(result_dict, model)` after `wandb.finish()` so you can run follow-up evaluations (used in the extrapolation cell for multiple test lengths without re-training).

In [11]:
# Cell 8 — Single experiment with W&B logging
def run_single_experiment(
    base_cfg, train_ids, valid_ids, attention_type, context_len, batch_size, return_model=False, **overrides
):
    """Run one experiment. Pass optional config overrides as keyword args, e.g.
    positional_encoding_type=\"rope\", block_type=\"conv_before_attention\", n_heads=16, n_kv_heads=2.
    """
    exp_cfg = copy.deepcopy(base_cfg)
    exp_cfg.attention_type = attention_type
    exp_cfg.context_len = context_len
    exp_cfg.batch_size = batch_size
    for key, val in overrides.items():
        if hasattr(exp_cfg, key):
            setattr(exp_cfg, key, val)
    set_seed(exp_cfg.seed)

    train_loader, valid_loader = build_dataloaders(train_ids, valid_ids, context_len, batch_size)
    model = TransformerLM(exp_cfg).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=exp_cfg.lr, weight_decay=exp_cfg.weight_decay)

    amp_enabled = USING_CUDA and bool(getattr(exp_cfg, "use_amp", False))
    amp_dtype = torch.bfloat16 if getattr(exp_cfg, "amp_dtype", "fp16") == "bf16" else torch.float16
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled and amp_dtype == torch.float16)

    run_name = (
        f"{attention_type}_ctx{context_len}_bs{batch_size}_"
        f"{exp_cfg.positional_encoding_type}_{exp_cfg.block_type}_h{exp_cfg.n_heads}_kv{exp_cfg.n_kv_heads}"
    )
    extra_wb_tags = list(globals().get("WANDB_EXTRA_TAGS", []) or [])
    wandb_init_kw = dict(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=run_name,
        config=asdict(exp_cfg),
        tags=[
            "attention-variants",
            "wikitext2",
            f"ctx-{context_len}",
            attention_type,
            exp_cfg.positional_encoding_type,
            exp_cfg.block_type,
        ]
        + extra_wb_tags,
        reinit="finish_previous",
    )
    wb_grp = globals().get("WANDB_GROUP")
    if wb_grp:
        wandb_init_kw["group"] = wb_grp
    run = wandb.init(**wandb_init_kw)

    n_params = sum(p.numel() for p in model.parameters())
    wandb.config.update({
        "num_parameters": n_params,
        "device": str(device),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "amp_enabled": amp_enabled,
        "amp_dtype": "bf16" if amp_dtype == torch.bfloat16 else "fp16",
    }, allow_val_change=True)

    hydra_omega = globals().get("_HYDRA_OMEGA")
    if hydra_omega is not None:
        from omegaconf import OmegaConf as _OC

        wandb.config.update(
            {
                "hydra_yaml": _OC.to_yaml(hydra_omega),
                "hydra": _OC.to_container(hydra_omega, resolve=True),
            },
            allow_val_change=True,
        )

    safe_cuda_empty_cache(reset_peak=True)

    nan_steps = 0
    grad_norms = []
    step_losses = []
    total_tokens = 0
    global_step = 0
    t0 = time.time()

    for epoch in range(exp_cfg.epochs):
        model.train()
        pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"{attention_type} ctx={context_len} ep={epoch+1}")
        for _, (x, y) in pbar:
            global_step += 1
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            amp_ctx = torch.cuda.amp.autocast(dtype=amp_dtype) if amp_enabled else nullcontext()
            with amp_ctx:
                _, loss = model(x, y)

            if torch.isnan(loss):
                nan_steps += 1
                continue

            if scaler.is_enabled():
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), exp_cfg.grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), exp_cfg.grad_clip)
                optimizer.step()

            loss_val = float(loss.item())
            grad_norm_val = float(grad_norm)
            step_losses.append(loss_val)
            grad_norms.append(grad_norm_val)
            total_tokens += x.numel()

            wandb.log({
                "train/step_loss": loss_val,
                "train/grad_norm": grad_norm_val,
                "train/lr": optimizer.param_groups[0]["lr"],
                "train/tokens_seen": total_tokens,
                "epoch": epoch + 1,
            }, step=global_step)

            if global_step % exp_cfg.eval_every_steps == 0:
                val_loss_i, val_ppl_i = evaluate(model, valid_loader, max_batches=exp_cfg.max_eval_batches)
                wandb.log({
                    "eval/intermediate_val_loss": val_loss_i,
                    "eval/intermediate_val_ppl": val_ppl_i,
                }, step=global_step)

            pbar.set_postfix({"loss": f"{loss_val:.4f}"})

    train_time = time.time() - t0
    train_tok_s = total_tokens / train_time if train_time > 0 else float("nan")
    train_time_per_epoch = train_time / max(exp_cfg.epochs, 1)

    val_loss, val_ppl = evaluate(model, valid_loader, max_batches=None)
    infer_tok_s, infer_ms_tok = inference_latency(model, exp_cfg.vocab_size, exp_cfg.context_len)
    peak_mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2) if device.type == "cuda" else float("nan")

    result = {
        "attention_type": attention_type,
        "positional_encoding_type": exp_cfg.positional_encoding_type,
        "block_type": exp_cfg.block_type,
        "n_heads": exp_cfg.n_heads,
        "n_kv_heads": exp_cfg.n_kv_heads,
        "context_len": context_len,
        "batch_size": batch_size,
        "num_parameters": n_params,
        "val_loss": val_loss,
        "val_ppl": val_ppl,
        "train_tokens_per_sec": train_tok_s,
        "train_time_sec_total": train_time,
        "train_time_sec_per_epoch": train_time_per_epoch,
        "infer_tokens_per_sec": infer_tok_s,
        "infer_ms_per_token": infer_ms_tok,
        "peak_gpu_mem_mb": peak_mem_mb,
        "nan_steps": nan_steps,
        "grad_norm_mean": float(np.mean(grad_norms)) if grad_norms else float("nan"),
        "grad_norm_max": float(np.max(grad_norms)) if grad_norms else float("nan"),
        "step_loss_std": float(np.std(step_losses)) if step_losses else float("nan"),
    }

    wandb.log({
        "eval/final_val_loss": result["val_loss"],
        "eval/final_val_ppl": result["val_ppl"],
        "system/train_time_sec_per_epoch": train_time_per_epoch,
        "system/train_tokens_per_sec": result["train_tokens_per_sec"],
        "system/infer_tokens_per_sec": result["infer_tokens_per_sec"],
        "system/infer_ms_per_token": result["infer_ms_per_token"],
        "system/peak_gpu_mem_mb": result["peak_gpu_mem_mb"],
        "stability/nan_steps": result["nan_steps"],
        "stability/grad_norm_mean": result["grad_norm_mean"],
        "stability/grad_norm_max": result["grad_norm_max"],
        "stability/step_loss_std": result["step_loss_std"],
    })
    wandb.summary.update(result)
    wandb.finish()

    if return_model:
        # Caller is responsible for cleanup when keeping model alive.
        return result, model

    cleanup_cuda(optimizer, train_loader, valid_loader, scaler, model)
    return result

## Explanation for Cell 9: Attention sweep + head/KV ablations + export

This driver covers **SAiDL Core ML §2** (attention variants + metrics) with a deliberate scope:

- **Contexts:** `512` and `1024` only — enough to show length effects while leaving budget for richer attention/head studies (skip 2048/4096 here; state that explicitly in your write-up).
- **Models:** Local window, sparse block, MQA, softmax-free ReLU, plus the softmax baseline. **`linear_performer`** is **not** in this loop (it used to dominate wall time); run the **optional cell right below** after re-executing the architecture cell — it logs to `attention_linear_performer_only.csv`. GQA head grid is still in this cell.
- **Head grid:** Extra runs vary `(n_heads, n_kv_heads)` for `gqa` to study grouped attention (§2 MQA/GQA bullet).

Outputs: CSV + LaTeX-friendly table with validation perplexity, throughput, latency, peak memory, gradient stats, **and** `train_time_sec_per_epoch` for § logging.

OOM handling uses `safe_cuda_empty_cache()` so a bad run does not wedge the CUDA context.

Next cells: **positional encodings** (§3), **conv hybrids** (§4), and **length extrapolation** (train 512 → val 512 & 1024).

In [12]:
# Cell 9 — Sweep + results export (512 / 1024 only; extra attention + head ablations instead of 2048/4096)

attention_variants = [
    "scaled_dot_product",
    "local_window",
    "sparse_block",
    "mqa",
    "softmax_free_relu",
]

context_lengths = [512, 1024]
batch_size_map = {512: 8, 1024: 4}

results = []

# Non-GQA attentions × contexts (uses cfg.n_heads / cfg.n_kv_heads where relevant)
for attn in attention_variants:
    for ctx in context_lengths:
        bs = batch_size_map[ctx]
        try:
            print(f"Running: attn={attn}, ctx={ctx}, bs={bs}")
            out = run_single_experiment(cfg, train_ids, valid_ids, attn, ctx, bs)
            results.append(out)
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"OOM at attn={attn}, ctx={ctx}. Skipping.")
                safe_cuda_empty_cache()
                results.append({"attention_type": attn, "context_len": ctx, "batch_size": bs, "status": "OOM"})
            else:
                raise

# GQA head-count / KV-head ablation (assignment §2 — MQA/GQA family), both context lengths
gqa_head_grid = [(8, 8), (8, 4), (8, 2), (8, 1)]
for ctx in context_lengths:
    bs = batch_size_map[ctx]
    for nh, nkv in gqa_head_grid:
        try:
            print(f"Running: attn=gqa, ctx={ctx}, heads={nh}, kv_heads={nkv}, bs={bs}")
            out = run_single_experiment(
                cfg, train_ids, valid_ids, "gqa", ctx, bs, n_heads=nh, n_kv_heads=nkv
            )
            results.append(out)
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"OOM gqa ctx={ctx} h={nh} kv={nkv}. Skipping.")
                safe_cuda_empty_cache()
                results.append(
                    {"attention_type": "gqa", "context_len": ctx, "n_heads": nh, "n_kv_heads": nkv, "status": "OOM"}
                )
            else:
                raise

df = pd.DataFrame(results)
display(df)
df.to_csv("attention_variant_results.csv", index=False)

report_df = df[df["status"].isna()].copy() if "status" in df.columns else df.copy()
keep_cols = [
    "attention_type",
    "n_heads",
    "n_kv_heads",
    "context_len",
    "batch_size",
    "val_ppl",
    "peak_gpu_mem_mb",
    "train_tokens_per_sec",
    "train_time_sec_per_epoch",
    "infer_tokens_per_sec",
    "nan_steps",
    "grad_norm_max",
]
keep_cols = [c for c in keep_cols if c in report_df.columns]
latex_table = report_df[keep_cols].sort_values([c for c in ["context_len", "attention_type", "n_heads"] if c in report_df.columns]).to_latex(index=False, float_format="%.4f")
print(latex_table)

Running: attn=scaled_dot_product, ctx=512, bs=8


C:\Users\sidsr\AppData\Local\Temp\ipykernel_14200\2382128706.py:23: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled and amp_dtype == torch.float16)


scaled_dot_product ctx=512 ep=1:   0%|          | 0/592 [00:00<?, ?it/s]C:\Users\sidsr\AppData\Local\Temp\ipykernel_14200\2382128706.py:88: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  amp_ctx = torch.cuda.amp.autocast(dtype=amp_dtype) if amp_enabled else nullcontext()
scaled_dot_product ctx=512 ep=1: 100%|██████████| 592/592 [02:32<00:00,  3.89it/s, loss=8.1258] 


epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/final_val_loss,▁
eval/final_val_ppl,▁
eval/intermediate_val_loss,█▁
eval/intermediate_val_ppl,█▁
stability/grad_norm_max,▁
stability/grad_norm_mean,▁
stability/nan_steps,▁
stability/step_loss_std,▁
system/infer_ms_per_token,▁
+8,...


Running: attn=scaled_dot_product, ctx=1024, bs=4


...

C:\Users\sidsr\AppData\Local\Temp\ipykernel_14200\2382128706.py:23: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled and amp_dtype == torch.float16)


gqa ctx=1024 ep=1:   0%|          | 0/592 [00:00<?, ?it/s]C:\Users\sidsr\AppData\Local\Temp\ipykernel_14200\2382128706.py:88: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  amp_ctx = torch.cuda.amp.autocast(dtype=amp_dtype) if amp_enabled else nullcontext()
gqa ctx=1024 ep=1: 100%|██████████| 592/592 [02:17<00:00,  4.29it/s, loss=8.8723] 


epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/final_val_loss,▁
eval/final_val_ppl,▁
eval/intermediate_val_loss,█▁
eval/intermediate_val_ppl,█▁
stability/nan_steps,▁
stability/step_loss_std,▁
system/infer_ms_per_token,▁
system/infer_tokens_per_sec,▁
system/peak_gpu_mem_mb,▁
+8,...


,attention_type,positional_encoding_type,block_type,n_heads,n_kv_heads,context_len,batch_size,num_parameters,val_loss,val_ppl,train_tokens_per_sec,train_time_sec_total,train_time_sec_per_epoch,infer_tokens_per_sec,infer_ms_per_token,peak_gpu_mem_mb,nan_steps,grad_norm_mean,grad_norm_max,step_loss_std
0,scaled_dot_product,learned,standard_transformer_block,8,2,512,8,55145984,8.493906,4884.911520,15918.014186,152.332569,152.332569,170.111333,5.878503,4682.776367,0,6.183403,126.478958,19.784663
1,scaled_dot_product,learned,standard_transformer_block,8,2,1024,4,55145984,8.623042,5558.270447,12222.125709,198.396912,198.396912,164.052339,6.095616,5582.229492,0,inf,inf,19.731172
2,local_window,learned,standard_transformer_block,8,2,512,8,55145984,8.436815,4613.835780,15593.284801,155.504888,155.504888,130.626614,7.655408,4680.041992,0,6.291668,125.477348,19.832004
3,local_window,learned,standard_transformer_block,8,2,1024,4,55145984,8.654846,5737.885388,12564.377466,192.992610,192.992610,146.428562,6.829269,5585.571289,0,6.456040,123.224945,19.781266
4,sparse_block,learned,standard_transformer_block,8,2,512,8,55145984,8.470248,4770.697462,14963.337270,162.051550,162.051550,100.185625,9.981472,4680.041992,0,6.213754,119.419235,19.824028
5,sparse_block,learned,standard_transformer_block,8,2,1024,4,55145984,8.686771,5924.022706,11629.755838,208.502400,208.502400,129.649137,7.713125,5583.571289,0,6.470746,121.730919,19.777646
6,mqa,learned,standard_transformer_block,8,2,512,8,51468800,8.588934,5371.881666,15670.638030,154.737286,154.737286,132.804633,7.529858,4685.379883,0,inf,inf,22.311575
7,mqa,learned,standard_transformer_block,8,2,1024,4,51468800,8.706761,6043.636989,5723.428335,423.667749,423.667749,87.790472,11.390758,5587.225586,0,inf,inf,22.274158
8,softmax_free_relu,learned,standard_transformer_block,8,2,512,8,55145984,8.501553,4922.406067,17349.566407,139.763262,139.763262,92.854242,10.769567,5451.041992,0,NaN,NaN,22.926270
9,softmax_free_relu,learned,standard_transformer_block,8,2,1024,4,55145984,8.571813,5280.693671,9760.727696,248.427379,248.427379,102.607793,9.745848,7121.305664,0,NaN,NaN,21.014027


\begin{tabular}{lrrrrrrrrrrr}
\toprule
attention_type & n_heads & n_kv_heads & context_len & batch_size & val_ppl & peak_gpu_mem_mb & train_tokens_per_sec & train_time_sec_per_epoch & infer_tokens_per_sec & nan_steps & grad_norm_max \\
\midrule
gqa & 8 & 8 & 512 & 8 & 4824.8010 & 4734.4502 & 22725.4979 & 106.7009 & 106.2078 & 0 & inf \\
gqa & 8 & 4 & 512 & 8 & 4951.8811 & 4706.4033 & 22794.7920 & 106.3766 & 109.7065 & 0 & 129.3070 \\
gqa & 8 & 2 & 512 & 8 & 4709.3028 & 4692.3799 & 23231.7769 & 104.3757 & 103.9707 & 0 & inf \\
gqa & 8 & 1 & 512 & 8 & 5371.8817 & 4685.3799 & 23360.0309 & 103.8026 & 99.7051 & 0 & inf \\
local_window & 8 & 2 & 512 & 8 & 4613.8358 & 4680.0420 & 15593.2848 & 155.5049 & 130.6266 & 0 & 125.4773 \\
mqa & 8 & 2 & 512 & 8 & 5371.8817 & 4685.3799 & 15670.6380 & 154.7373 & 132.8046 & 0 & inf \\
scaled_dot_product & 8 & 2 & 512 & 8 & 4884.9115 & 4682.7764 & 15918.0142 & 152.3326 & 170.1113 & 0 & 126.4790 \\
softmax_free_relu & 8 & 2 & 512 & 8 & 4922.4061 & 5451.0420

## Part 3 & 4 — Positional encodings + convolution–attention hybrids

**Assignment mapping**
- §3 Positional embeddings: compare `learned`, `sinusoidal`, `rope`, `alibi`, `relative_shaw` under a **fixed** attention type so differences come from position handling.
- §4 Conv + attention: compare `block_type` (`standard` vs causal conv hybrids) using your strongest attention + PE from earlier runs.

**How to configure**
- Set `POS_SWEEP_ATTN` to the attention string you want held constant (default softmax baseline).
- Adjust `positional_results = []` loops to subset encodings if runtime is tight.
- Hybrid section toggles `HYBRID_BLOCK_TYPES`; pair with `HYBRID_ATTN` / optional kwargs (`positional_encoding_type`, etc.).

CSV outputs are separate from Cell 9 so you can merge tables in your LaTeX report.

In [15]:
# Cell 10 — Positional sweep + conv hybrids (run after Cell 9 if you want separate CSVs)

# Runtime hotfix: patch RoPE forward in case kernel still holds an older class definition.
def _rope_forward_fixed(self, q, k):
    """q,k: (B, H, T, Dh); Dh must be even."""
    B, H, T, Dh = q.shape
    if Dh % 2 != 0:
        raise ValueError(f"RoPE requires even head dim, got Dh={Dh}")

    t = torch.arange(T, device=q.device, dtype=self.inv_freq.dtype)
    freqs = torch.outer(t, self.inv_freq)  # (T, Dh/2)
    cos = torch.repeat_interleave(freqs.cos(), repeats=2, dim=-1)[None, None, :, :]
    sin = torch.repeat_interleave(freqs.sin(), repeats=2, dim=-1)[None, None, :, :]
    q_embed = (q * cos) + (_rotate_half(q) * sin)
    k_embed = (k * cos) + (_rotate_half(k) * sin)
    return q_embed, k_embed

RotaryEmbedding.forward = _rope_forward_fixed

POS_SWEEP_ATTN = "scaled_dot_product"
POS_CONTEXT = 512
POS_BATCH = 8
positional_types = ["rope", "alibi", "relative_shaw"]

positional_rows = []
for pe in positional_types:
    try:
        print(f"[positional] attn={POS_SWEEP_ATTN}, pe={pe}, ctx={POS_CONTEXT}")
        positional_rows.append(
            run_single_experiment(
                cfg,
                train_ids,
                valid_ids,
                POS_SWEEP_ATTN,
                POS_CONTEXT,
                POS_BATCH,
                positional_encoding_type=pe,
                max_position_embeddings=max(cfg.max_position_embeddings, POS_CONTEXT, 1024),
            )
        )
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"OOM positional pe={pe}")
            positional_rows.append({"positional_encoding_type": pe, "status": "OOM"})
            safe_cuda_empty_cache()
        else:
            raise

df_pos = pd.DataFrame(positional_rows)
display(df_pos)
df_pos.to_csv("positional_encoding_results.csv", index=False)

# --- Conv + attention hybrids (§4): pick defaults or pin your best attn/PE here ---
HYBRID_ATTN = "gqa"
HYBRID_CTX = 512
HYBRID_BS = 8
HYBRID_KW = dict(n_heads=8, n_kv_heads=2, positional_encoding_type="rope")

HYBRID_BLOCK_TYPES = [
    "standard_transformer_block",
    "conv_before_attention",
    "interleaved_conv_attention",
    "alternate_attention_dwconv",
]

hybrid_rows = []
for bt in HYBRID_BLOCK_TYPES:
    try:
        print(f"[hybrid] block={bt}, attn={HYBRID_ATTN}")
        hybrid_rows.append(
            run_single_experiment(
                cfg,
                train_ids,
                valid_ids,
                HYBRID_ATTN,
                HYBRID_CTX,
                HYBRID_BS,
                block_type=bt,
                **HYBRID_KW,
            )
        )
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            hybrid_rows.append({"block_type": bt, "status": "OOM"})
            safe_cuda_empty_cache()
        else:
            raise

df_hybrid = pd.DataFrame(hybrid_rows)
display(df_hybrid)
df_hybrid.to_csv("conv_hybrid_results.csv", index=False)

[positional] attn=scaled_dot_product, pe=rope, ctx=512


C:\Users\sidsr\AppData\Local\Temp\ipykernel_14200\2382128706.py:23: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled and amp_dtype == torch.float16)


scaled_dot_product ctx=512 ep=1:   0%|          | 0/592 [00:00<?, ?it/s]C:\Users\sidsr\AppData\Local\Temp\ipykernel_14200\2382128706.py:88: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  amp_ctx = torch.cuda.amp.autocast(dtype=amp_dtype) if amp_enabled else nullcontext()
scaled_dot_product ctx=512 ep=1: 100%|██████████| 592/592 [02:44<00:00,  3.59it/s, loss=8.2451] 


epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/final_val_loss,▁
eval/final_val_ppl,▁
eval/intermediate_val_loss,█▁
eval/intermediate_val_ppl,█▁
stability/nan_steps,▁
stability/step_loss_std,▁
system/infer_ms_per_token,▁
system/infer_tokens_per_sec,▁
system/peak_gpu_mem_mb,▁
+8,...


[positional] attn=scaled_dot_product, pe=alibi, ctx=512


...

[hybrid] block=alternate_attention_dwconv, attn=gqa


C:\Users\sidsr\AppData\Local\Temp\ipykernel_14200\2382128706.py:23: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled and amp_dtype == torch.float16)


gqa ctx=512 ep=1:   0%|          | 0/592 [00:00<?, ?it/s]C:\Users\sidsr\AppData\Local\Temp\ipykernel_14200\2382128706.py:88: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  amp_ctx = torch.cuda.amp.autocast(dtype=amp_dtype) if amp_enabled else nullcontext()
gqa ctx=512 ep=1: 100%|██████████| 592/592 [01:28<00:00,  6.67it/s, loss=8.2814] 


epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/final_val_loss,▁
eval/final_val_ppl,▁
eval/intermediate_val_loss,█▁
eval/intermediate_val_ppl,█▁
stability/nan_steps,▁
stability/step_loss_std,▁
system/infer_ms_per_token,▁
system/infer_tokens_per_sec,▁
system/peak_gpu_mem_mb,▁
+8,...


,attention_type,positional_encoding_type,block_type,n_heads,n_kv_heads,context_len,batch_size,num_parameters,val_loss,val_ppl,train_tokens_per_sec,train_time_sec_total,train_time_sec_per_epoch,infer_tokens_per_sec,infer_ms_per_token,peak_gpu_mem_mb,nan_steps,grad_norm_mean,grad_norm_max,step_loss_std
0,gqa,rope,standard_transformer_block,8,2,512,8,47799808,8.436257,4611.263719,22630.236236,107.150097,107.150097,71.363882,14.012691,4878.715820,0,inf,inf,27.196163
1,gqa,rope,conv_before_attention,8,2,512,8,62492160,8.290778,3986.935033,19280.565990,125.765603,125.765603,45.809288,21.829634,5194.436523,0,inf,inf,16.129322
2,gqa,rope,interleaved_conv_attention,8,2,512,8,55145984,8.428800,4577.006355,20666.435852,117.331891,117.331891,62.138438,16.093098,5037.303711,0,inf,inf,18.208379
3,gqa,rope,alternate_attention_dwconv,8,2,512,8,46240256,8.284056,3960.224591,27332.388852,88.716431,88.716431,108.149377,9.246470,4328.520508,0,inf,inf,22.856193


## Extrapolation test (§3c) — train at L=512, validate at L ∈ {512, 1024}

The official brief also mentions L_test=2048; we skip 2048 here to match the **512/1024-focused** experiment plan—note that in your report.

**Procedure**
1. Train with `context_len=512` and `max_position_embeddings ≥ 1024` (required for learned/sinusoidal tables; RoPE/ALiBi/Shaw extrapolate naturally up to max buffer).
2. Reuse the trained `nn.Module` (`return_model=True`) and evaluate with `build_valid_loader` + `evaluate` at each test length **without** extra training steps.

**Configure** `EXTRAP_ATTN` and `EXTRAP_PE_LIST` to sweep which positional families you want to compare under a fixed attention mechanism.

In [16]:
# Cell 11 — Length extrapolation (512 train → val 512 & 1024)


@torch.no_grad()
def val_ppl_at_context(model, valid_ids, context_len, batch_size, max_batches=None):
    loader = build_valid_loader(valid_ids, context_len, batch_size)
    loss, ppl = evaluate(model, loader, max_batches=max_batches)
    return loss, ppl


EXTRAP_TRAIN_CTX = 512
EXTRAP_EVAL_CTX = [512, 1024]
EXTRAP_BS = 8
EXTRAP_ATTN = "scaled_dot_product"
EXTRAP_PE_LIST = ["rope", "alibi", "relative_shaw", "learned", "sinusoidal"]

extrap_rows = []
for pe in EXTRAP_PE_LIST:
    try:
        pair = run_single_experiment(
            cfg,
            train_ids,
            valid_ids,
            EXTRAP_ATTN,
            EXTRAP_TRAIN_CTX,
            EXTRAP_BS,
            return_model=True,
            positional_encoding_type=pe,
            max_position_embeddings=max(8192, max(EXTRAP_EVAL_CTX)),
        )
        res_train, model = pair
        for L in EXTRAP_EVAL_CTX:
            loss_l, ppl_l = val_ppl_at_context(
                model, valid_ids, L, min(EXTRAP_BS, 4), max_batches=cfg.max_eval_batches
            )
            extrap_rows.append(
                {
                    "positional_encoding_type": pe,
                    "attention_type": EXTRAP_ATTN,
                    "train_ctx": EXTRAP_TRAIN_CTX,
                    "eval_ctx": L,
                    "val_loss": loss_l,
                    "val_ppl": ppl_l,
                }
            )
        del model
        safe_cuda_empty_cache()
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            extrap_rows.append({"positional_encoding_type": pe, "status": "OOM"})
            safe_cuda_empty_cache()
        else:
            raise

df_extrap = pd.DataFrame(extrap_rows)
display(df_extrap)
df_extrap.to_csv("positional_extrapolation_512_train.csv", index=False)

C:\Users\sidsr\AppData\Local\Temp\ipykernel_14200\2382128706.py:23: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled and amp_dtype == torch.float16)


scaled_dot_product ctx=512 ep=1:   0%|          | 0/592 [00:00<?, ?it/s]C:\Users\sidsr\AppData\Local\Temp\ipykernel_14200\2382128706.py:88: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  amp_ctx = torch.cuda.amp.autocast(dtype=amp_dtype) if amp_enabled else nullcontext()
scaled_dot_product ctx=512 ep=1: 100%|██████████| 592/592 [01:49<00:00,  5.39it/s, loss=8.2451] 


epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/final_val_loss,▁
eval/final_val_ppl,▁
eval/intermediate_val_loss,█▁
eval/intermediate_val_ppl,█▁
stability/nan_steps,▁
stability/step_loss_std,▁
system/infer_ms_per_token,▁
system/infer_tokens_per_sec,▁
system/peak_gpu_mem_mb,▁
+8,...


C:\Users\sidsr\AppData\Local\Temp\ipykernel_14200\2382128706.py:23: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled and amp_dtype == torch.float16)


scaled_dot_product ctx=512 ep=1:   0%|          | 0/592 [00:00<?, ?it/s]C:\Users\sidsr\AppData\Local\Temp\ipykernel_14200\2382128706.py:88: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  amp_ctx = torch.cuda.amp.autocast(dtype=amp_dtype) if amp_enabled else nullcontext()
scaled_dot_product ctx=512 ep=1: 100%|██████████| 592/592 [01:52<00:00,  5.28it/s, loss=8.3465] 


...

epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/final_val_loss,▁
eval/final_val_ppl,▁
eval/intermediate_val_loss,█▁
eval/intermediate_val_ppl,█▁
stability/grad_norm_max,▁
stability/grad_norm_mean,▁
stability/nan_steps,▁
stability/step_loss_std,▁
system/infer_ms_per_token,▁
+8,...


C:\Users\sidsr\AppData\Local\Temp\ipykernel_14200\2382128706.py:23: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled and amp_dtype == torch.float16)


scaled_dot_product ctx=512 ep=1:   0%|          | 0/592 [00:00<?, ?it/s]C:\Users\sidsr\AppData\Local\Temp\ipykernel_14200\2382128706.py:88: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  amp_ctx = torch.cuda.amp.autocast(dtype=amp_dtype) if amp_enabled else nullcontext()
scaled_dot_product ctx=512 ep=1: 100%|██████████| 592/592 [01:44<00:00,  5.67it/s, loss=8.6336] 


epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/final_val_loss,▁
eval/final_val_ppl,▁
eval/intermediate_val_loss,█▁
eval/intermediate_val_ppl,█▁
stability/grad_norm_max,▁
stability/grad_norm_mean,▁
stability/nan_steps,▁
stability/step_loss_std,▁
system/infer_ms_per_token,▁
+8,...


,positional_encoding_type,attention_type,train_ctx,eval_ctx,val_loss,val_ppl
0,rope,scaled_dot_product,512,512,8.294667,4002.469270
1,rope,scaled_dot_product,512,1024,8.367356,4304.241025
2,alibi,scaled_dot_product,512,512,8.429672,4580.999520
3,alibi,scaled_dot_product,512,1024,8.473143,4784.529679
4,relative_shaw,scaled_dot_product,512,512,8.288250,3976.869844
5,relative_shaw,scaled_dot_product,512,1024,8.347508,4219.653716
6,learned,scaled_dot_product,512,512,8.461144,4727.462285
7,learned,scaled_dot_product,512,1024,8.521251,5020.329422
8,sinusoidal,scaled_dot_product,512,512,8.403995,4464.870268
9,sinusoidal,scaled_dot_product,512,1024,8.483488,4834.282753
